# B5 + B6: Focal loss + Augmentation + SupCon cho M1/M2 DiffMul

- **B5**: focal BCE (gamma=2, alpha=0.5) giảm áp lực easy negatives.
- **B6**: xóa comment + đổi tên identifier nhất quán (giữ nhãn) + SupCon trên vector CLS.

Huấn luyện lại 2 mô hình từ đầu với protocol giữ nguyên (30k train / 3k val / 5k test).


In [ ]:
import base64, io, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

WORK = Path('/kaggle/working')
CLEAN = WORK / 'clean_b5b6'
CLEAN.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print(f'\n$ {cmd}', flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with (WORK / 'live_progress.log').open('a', encoding='utf-8') as lf:
        for line in iter(proc.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'Lenh that bai (rc={rc}): {cmd}')


In [ ]:
payload = '...'
repo_root = WORK / 'project'
repo_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as tar:
    tar.extractall(repo_root)
os.chdir(repo_root)
print('Repo:', os.getcwd(), flush=True)
sh('ls configs/tuning/exp_m1_b5b6.yaml configs/tuning/exp_m2_b5b6.yaml')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise SystemError('Kernel chua bat GPU!')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
sh('pip install -q -U transformers datasets scikit-learn scipy pyarrow \"tree-sitter>=0.23.0\" \"tree-sitter-java>=0.23.5\"')

In [ ]:
data_raw = repo_root / 'data/raw/bigclonebench'
data_raw.mkdir(parents=True, exist_ok=True)
for name in ('bcb_train.parquet', 'bcb_validation.parquet', 'bcb_test.parquet'):
    cands = list(Path('/kaggle/input').rglob(name))
    if cands and not (data_raw / name).exists():
        sh(f'cp "{cands[0]}" "{data_raw / name}"')
    print(name, 'exists=', (data_raw / name).exists(), flush=True)
if any(not (data_raw / n).exists() for n in ('bcb_train.parquet', 'bcb_validation.parquet', 'bcb_test.parquet')):
    print('Tai lai splits tu Hugging Face...', flush=True)
    sh('python scripts/download_bcb_subsets.py --splits-custom --train-samples 30000 --val-samples 3000 --test-samples 5000')
for n in ('bcb_train.parquet', 'bcb_validation.parquet', 'bcb_test.parquet'):
    assert (data_raw / n).exists(), f'Thieu {n}'

In [ ]:
# HUẤN LUYỆN M1 VỚI B5 + B6
print('=' * 70)
print('>>> BAT DAU M1: focal + augmentation + SupCon <<<')
print('=' * 70)
cfg = 'configs/tuning/exp_m1_b5b6.yaml'
t_start = time.perf_counter()
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode preflight --device cuda')
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode train --device cuda --time-budget-s 18000')
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode evaluate --device cuda')
print(f'>>> HOAN TAT M1 TRONG {(time.perf_counter()-t_start)/60:.1f} PHUT!', flush=True)

# Backup ngay kết quả trước khi chạy model tiếp theo
res_dir = repo_root / 'results' / 'exp_m1_b5b6'
if res_dir.exists():
    for p in res_dir.glob('*'):
        if p.is_file() and p.suffix in ('.pt', '.json'):
            name = 'exp_m1_b5b6_' + p.name
            shutil.copy2(p, CLEAN / name)
            shutil.copy2(p, WORK / name)
            print(f'  [SAVED] {name}', flush=True)
else:
    print('Canh bao: khong thay res_dir', flush=True)

In [ ]:
# HUẤN LUYỆN M2 VỚI B5 + B6
print('=' * 70)
print('>>> BAT DAU M2: focal + augmentation + SupCon <<<')
print('=' * 70)
cfg = 'configs/tuning/exp_m2_b5b6.yaml'
t_start = time.perf_counter()
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode preflight --device cuda')
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode train --device cuda --time-budget-s 18000')
sh(f'python -m src.experiments.bcb_runner --config {cfg} --mode evaluate --device cuda')
print(f'>>> HOAN TAT M2 TRONG {(time.perf_counter()-t_start)/60:.1f} PHUT!', flush=True)

# Backup ngay kết quả trước khi chạy model tiếp theo
res_dir = repo_root / 'results' / 'exp_m2_b5b6'
if res_dir.exists():
    for p in res_dir.glob('*'):
        if p.is_file() and p.suffix in ('.pt', '.json'):
            name = 'exp_m2_b5b6_' + p.name
            shutil.copy2(p, CLEAN / name)
            shutil.copy2(p, WORK / name)
            print(f'  [SAVED] {name}', flush=True)
else:
    print('Canh bao: khong thay res_dir', flush=True)

In [ ]:
print('=' * 70)
print('KET QUA TEST:', flush=True)
for name in ('exp_m1_b5b6', 'exp_m2_b5b6'):
    p = repo_root / 'results' / name / 'test_metrics.json'
    print(name, p.read_text(encoding='utf-8') if p.exists() else 'MISSING', flush=True)

with tarfile.open(WORK / 'b5b6_bundle.tar.gz', 'w:gz') as tar:
    for f in CLEAN.glob('*'):
        tar.add(f, arcname=f.name)
print('>>> Bundle san sang.', flush=True)
os.chdir(WORK)
for junk in ('project', 'clean_b5b6'):
    shutil.rmtree(WORK / junk, ignore_errors=True)
print('>>> XONG 100%!', flush=True)